In [1]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
np.random.seed(42)

QUESTION 1

In [3]:

A = np.random.normal(loc=50, scale=12, size=(1000,5))
print(A.shape, A.dtype, A.nbytes)     # (1000, 5) float64 40000
print(A.astype(np.float32).nbytes)    # 20000

(1000, 5) float64 40000
20000


QUESTION 2

In [7]:
#q2
Z =(A-A.mean(axis=0)) / A.std(axis=0)
print(np.round(Z.mean(axis=0),6))
print(np.round(Z.std(axis=0),6))

[-0. -0.  0. -0. -0.]
[1. 1. 1. 1. 1.]


QUESTION 3

In [8]:
#q3
def minmax(X, axis):
    mn = X.min(axis=axis, keepdims=True)
    mx = X.max(axis=axis, keepdims=True)
    return (X - mn) / (mx - mn)

# Test on axis=0 (each column scaled)
M0 = minmax(A, axis=0)
print("axis=0 min:", M0.min(axis=0))
print("axis=0 max:", M0.max(axis=0))

# Test on axis=1 (each row scaled)
M1 = minmax(A, axis=1)
print("axis=1 min (first 5 rows):", M1.min(axis=1)[:5])
print("axis=1 max (first 5 rows):", M1.max(axis=1)[:5])

axis=0 min: [0. 0. 0. 0. 0.]
axis=0 max: [1. 1. 1. 1. 1.]
axis=1 min (first 5 rows): [0. 0. 0. 0. 0.]
axis=1 max (first 5 rows): [1. 1. 1. 1. 1.]


QUESTION 4

In [9]:
#q4
# Pick exactly 5% of all positions at random
n_nan = int(0.05 * A.size)                          # 250 positions
flat_idx = np.random.choice(A.size, n_nan, replace=False)

nan_mask = np.zeros(A.size, dtype=bool)
nan_mask[flat_idx] = True
nan_mask = nan_mask.reshape(A.shape)                # (1000, 5) True/False grid

A[nan_mask] = np.nan
print("NaNs injected:", np.isnan(A).sum())          # 250

# Fill each NaN with the mean of its own column
col_means = np.nanmean(A, axis=0)                   # ignores NaNs, shape (5,)
A[nan_mask] = np.broadcast_to(col_means, A.shape)[nan_mask]

print(np.isnan(A).sum() == 0)                       # True

NaNs injected: 250
True


QUESTION 5

In [10]:
#q5
mu = A.mean(axis=0)
sd = A.std(axis=0)

outlier_mask = np.abs(A - mu) > 3 * sd              # True where |value - mean| > 3 sd
med = np.median(A, axis=0)                          # median of each column

print("Replaced per column:", outlier_mask.sum(axis=0))

A = np.where(outlier_mask, med, A)

Replaced per column: [5 1 5 3 5]


QUESTION 6

In [11]:
#q6
X = A[:200]                                         # first 200 rows, shape (200, 5)

diff = X[:, None, :] - X[None, :, :]                # (200, 200, 5)
D = np.sqrt((diff ** 2).sum(axis=-1))               # (200, 200)

print(D.shape)
print("Diagonal is 0:", np.allclose(np.diag(D), 0))
print("Symmetric:", np.allclose(D, D.T))

(200, 200)
Diagonal is 0: True
Symmetric: True


QUESTION 7

In [13]:
#q7
D2 = D.copy()
np.fill_diagonal(D2, np.inf)                 # so a row is never its own neighbour

nn_idx = np.argpartition(D2, 5, axis=1)[:, :5]
print(nn_idx.shape, nn_idx.dtype)            # (200, 5) int64
print(nn_idx[:3])

(200, 5) int64
[[180 179 116 178 192]
 [114  17  31 112 197]
 [  3 158 132 145 102]]


QUESTION 8

In [14]:
#q8
arr = np.random.randint(0, 100, size=1_000_000)

In [15]:
%%time
import time
t0 = time.perf_counter()
counts_np = np.bincount(arr, minlength=100)
t_np = time.perf_counter() - t0

CPU times: user 3.15 ms, sys: 0 ns, total: 3.15 ms
Wall time: 2.59 ms


In [16]:
%%time
import time
t0 = time.perf_counter()
counts_py = [0] * 100
for v in arr.tolist():
    counts_py[v] += 1
t_py = time.perf_counter() - t0

CPU times: user 123 ms, sys: 1.84 ms, total: 125 ms
Wall time: 126 ms


In [17]:
print("Same result:", np.array_equal(counts_np, counts_py))
print(f"Speed-up: {t_py / t_np:.1f}x")

Same result: True
Speed-up: 49.0x


QUESTION 9

In [18]:
x = np.random.randn(10000)

windows = sliding_window_view(x, 10)
print(windows.shape)                          # (9991, 10)

win_mean = windows.mean(axis=1)

ref = pd.Series(x).rolling(10).mean().to_numpy()[9:]   # drop the first 9 NaNs
print(np.allclose(win_mean, ref))             # True

(9991, 10)
True


QUESTION 10

In [19]:
B = np.random.randn(12, 8)

R = B.reshape(4, 3, 8)
out = R.mean(axis=1)
print(R.shape, out.shape)                     # (4, 3, 8) (4, 8)

(4, 3, 8) (4, 8)


QUESTION 11

In [20]:
np.save("clean.npy", A)
A_loaded = np.load("clean.npy")

assert np.array_equal(A, A_loaded)
print("Loaded array is identical to the original")

Loaded array is identical to the original
